# FPP15 — Final Statistics, Figures and Report

## Contents
1. **Main time comparison — image-embedding matching (ours) vs FingerJetFX + BOZORTH3 matching** (daily 1:1 login), re-timed in **one session on one machine**: our image model on the T4 (plain, **XLA-compiled**, XLA + fp16) and on the CPU; FingerJetFX + BOZORTH3 on the CPU (1 thread, as on the terminal). XLA fuses the network's layers into few GPU kernels — the same idea as an NPU compiler — so it is the closest Colab proxy for a terminal NPU running one fingerprint at a time. Warm-up, loading, tracing / compilation and decoding are excluded.
2. **Main accuracy comparison** on the identical pairs: image vs image (our final image model), FingerJetFX + BOZORTH3, the minutiae encoder, and the first-login migration match — validation, test, DB1 / DB2 / DB1×DB2 and FingerDOS, all paper metrics, mean ± std over 3 repetitions, plus paired differences vs BOZORTH3.
3. **Progression** of every kernel's winner (image branch, minutiae branch, migration).
4. Reported only (from FPP14): first login, migration time, complexity.
5. Figures, workbook, CSVs and a Markdown summary.

**Runtime:** Colab **T4 GPU** for section 1 (results are cached: later runs only rebuild the report unless `RERUN_TIMING = True`). **Needs:** all kernel outputs FPP03 – FPP14, fpp_lib ≥ 1.13.

In [ ]:
%pip install -q openpyxl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load the shared libraries from output/00_code/ (fpf_lib from FPF00, fpp_lib installed by FPP01), so every FPP kernel uses the same version.
import shutil, sys, importlib
from pathlib import Path
def _find_folder(name, root=Path('/content/drive/MyDrive'), max_depth=4):
    frontier = [root]
    for _ in range(max_depth + 1):
        nxt = []
        for d in frontier:
            try: kids = [c for c in d.iterdir() if c.is_dir()]
            except OSError: continue
            for c in kids:
                if c.name == name: return c
                nxt.append(c)
        frontier = nxt
    return None
ROOT = _find_folder('FingerPrint-Final')
if ROOT is None: raise FileNotFoundError("'FingerPrint-Final' not found under My Drive")
CODE = ROOT / 'output' / '00_code'
for m in ('fpf_lib.py', 'fpf_tf.py', 'fpp_lib.py'):
    src = CODE / m
    if not src.is_file(): raise FileNotFoundError(f'{src} missing: run ' + ('FPP01' if m == 'fpp_lib.py' else 'FPF00') + ' first')
    shutil.copy2(src, Path('/content') / m)
sys.path.insert(0, '/content')
import fpf_lib as L; importlib.reload(L)
import fpf_tf as T; importlib.reload(T)
import fpp_lib as P; importlib.reload(P)
import numpy as np, pandas as pd, json, time, os, gc
if tuple(int(x) for x in P.VERSION.split('.')) < (1, 13): raise RuntimeError(f'fpp_lib {P.VERSION} in output/00_code is too old: upload fpp_lib 1.13')
ENV = L.env_info(); print('fpp_lib', P.VERSION, '|', json.dumps(ENV, indent=1))

## 1. Configuration

In [ ]:
import subprocess, shutil, matplotlib.pyplot as plt
KERNEL = 'FPP15'; ws = L.workspace(); KDIR = P.kernel_dir(ws, KERNEL); FIG = KDIR / 'figures'; FIG.mkdir(parents=True, exist_ok=True)
TDIR = KDIR / 'timing'; TDIR.mkdir(exist_ok=True); WORK = Path('/content/fpp15'); WORK.mkdir(exist_ok=True)
REPS = L.repetitions(ws); REP = REPS[0]; RERUN_TIMING = False
N_ITEMS = 300; WARMUP = 30; MEASURE_REPEATS = 3; BATCH = 64
def winner(kernel):
    p_ = P.kernel_dir(ws, kernel) / 'winner_manifest.json'
    if not p_.is_file(): return None, None
    mf = json.loads(p_.read_text())
    if 'files' in mf: return {r: Path(ws['output']) / f['pair_scores.npz'] for r, f in mf['files'].items()}, mf
    if 'winning_arm' in mf: return {r: P.kernel_dir(ws, kernel) / 'runs' / r / mf['winning_arm'] / 'pair_scores.npz' for r in L.repetitions(ws)}, mf          # older manifests (FPP04 / FPP04C)
    return None, mf
img_files, mf5 = winner('FPP05'); mig_files, mf13 = winner('FPP13'); _, mf9 = winner('FPP09')
IMG_DIR = img_files[REP].parent; MIG_DIR = mig_files[REP].parent
GPU = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).returncode == 0
GPU_NAME = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() if GPU else 'none'
SPEC = dict(rep=REP, img_dir=str(IMG_DIR), image_model=json.loads((IMG_DIR / 'encoder_config.json').read_text()).get('backbone', 'EfficientNetV2S') if (IMG_DIR / 'encoder_config.json').is_file() else 'EfficientNetV2S',
            mig_dir=str(MIG_DIR), module=str(P.kernel_dir(ws, 'FPP13') / 'fpp13_minutiae.py'), minutiae_config=mf13.get('minutiae_config', mf9['config']), n_items=N_ITEMS, warmup=WARMUP, batch=BATCH)
(WORK / 'spec.json').write_text(json.dumps(SPEC, indent=1))
CONFIGS = ([dict(device='gpu', precision='fp32', threads=0, jit='plain'), dict(device='gpu', precision='fp32', threads=0, jit='xla'), dict(device='gpu', precision='fp16', threads=0, jit='xla')] if GPU else []) + \
          [dict(device='cpu', precision='fp32', threads=1, jit='plain'), dict(device='cpu', precision='fp32', threads=1, jit='xla'), dict(device='cpu', precision='fp32', threads=os.cpu_count(), jit='plain')]
IMG_LABEL = 'Ours: image vs image (FPP05 winner)'; BOZ = 'FingerJetFX + BOZORTH3 (FPP03)'; MIN_LABEL = 'Minutiae encoder (FPP09 winner)'; MIG_LABEL = 'First login after migration (FPP13 winner)'
CONFIG = dict(kernel=KERNEL, kernel_name=P.FPP_KERNELS[KERNEL], repetition_timed=REP, image_model=L.rel(ws, IMG_DIR), gpu=GPU_NAME, configs=CONFIGS, items=N_ITEMS, warmup_excluded=WARMUP,
              measure_repeats=MEASURE_REPEATS, rerun_timing=RERUN_TIMING, fpp_lib_version=P.VERSION)
print(json.dumps(CONFIG, indent=2, default=str))

## 2. Timing workers (fresh process per configuration; the image model is checked against its saved embeddings before timing)

In [ ]:
WORKER = r"""
import os, sys, json, time, numpy as np
device, precision, threads, out = sys.argv[1], sys.argv[2], int(sys.argv[3]), sys.argv[4]; jit = len(sys.argv) > 5 and sys.argv[5] == 'xla'
if device == 'cpu':
    os.environ['CUDA_VISIBLE_DEVICES'] = ''
    for v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'): os.environ[v] = str(threads)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
if device == 'cpu': tf.config.threading.set_intra_op_parallelism_threads(threads); tf.config.threading.set_inter_op_parallelism_threads(threads)
else:
    for g in tf.config.list_physical_devices('GPU'): tf.config.experimental.set_memory_growth(g, True)
sys.path.insert(0, '/content'); import matplotlib; matplotlib.use('Agg'); import fpf_lib as L, h5py, importlib.util
S = json.load(open('/content/fpp15/spec.json')); ws = L.workspace(); IMAGE_SIZE = 224; EMBEDDING_DIM = 1024; DROPOUT = 0.30
spec = importlib.util.spec_from_file_location('fpp13_minutiae', S['module']); M = importlib.util.module_from_spec(spec); spec.loader.exec_module(M)
M_CFG = dict(M.SETUP, **S['minutiae_config']); Lyr = tf.keras.layers
# ---- migration matcher (always fp32): FPP13 minutiae encoder + Siamese tower
class ModalityOffset(Lyr.Layer):
    def build(self, shape): self.b = self.add_weight(name='offset', shape=(int(shape[-1]),), initializer='zeros')
    def call(self, x): return x + self.b
def build_siamese():
    xm = tf.keras.Input((M.EMBEDDING_DIM,), name='minutiae_embedding'); xi = tf.keras.Input((M.EMBEDDING_DIM,), name='image_embedding')
    l2 = lambda n: Lyr.Lambda(lambda z: tf.math.l2_normalize(z, axis=1), name=n)
    om = ModalityOffset(name='minutiae_offset'); oi = ModalityOffset(name='image_offset')
    hd = Lyr.Dense(2048, activation='gelu', name='shared_hidden'); dr = Lyr.Dropout(0.1, name='shared_dropout'); out_ = Lyr.Dense(M.EMBEDDING_DIM, name='shared_out')
    f = tf.keras.Model(xm, l2('minutiae_shared_embedding')(out_(dr(hd(om(xm))))), name='siamese_minutiae'); g = tf.keras.Model(xi, l2('image_shared_embedding')(out_(dr(hd(oi(xi))))), name='siamese_image')
    return f, g
md_ = S['mig_dir']; enc = M.build_ls_encoder(M_CFG); enc.load_weights(md_ + '/encoder.weights.h5')
ft, gt = build_siamese(); ft.load_weights(md_ + '/minutiae_tower.weights.h5'); gt.load_weights(md_ + '/image_tower.weights.h5')
MINU = M.load_minutiae(L.local_copy(ws['templates'] / 'MOLF_DB1_fingerjetfx_templates.h5'))
SETS = np.stack([M.to_set(m, quality=M_CFG['quality']) for m in MINU[:S['n_items']]])
# ---- image model (FPP04C / FPP04L architecture), fp32 or mixed fp16
if precision == 'fp16': tf.keras.mixed_precision.set_global_policy('mixed_float16')
A = tf.keras.applications
def build_image_model(name):
    bb = getattr(A, name)(include_top=False, weights=None, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), pooling='avg')
    image = tf.keras.Input((IMAGE_SIZE, IMAGE_SIZE, 1), dtype=tf.uint8, name='fingerprint_uint8')
    x = Lyr.Lambda(lambda z: tf.cast(z, tf.float32), name='cast_float')(image); x = Lyr.Concatenate(name='gray_to_rgb')([x, x, x])
    f = bb(x, training=False); f = Lyr.Dropout(DROPOUT, name='head_dropout')(f)
    e = Lyr.Dense(EMBEDDING_DIM, use_bias=False, name='embedding_dense')(f); e = Lyr.BatchNormalization(name='embedding_bn')(e)
    e = Lyr.Lambda(lambda z: tf.math.l2_normalize(tf.cast(z, tf.float32), axis=1), name='embedding')(e)
    return tf.keras.Model(image, e, name=f'{name}_full_encoder')
img = build_image_model(S['image_model']); img.load_weights(S['img_dir'] + '/encoder.weights.h5')
X = []
for d in ('MOLF_DB1', 'MOLF_DB2', 'FingerDOS'):
    with h5py.File(L.local_copy(ws['images'] / f'{d}_standardized_grayscale_{IMAGE_SIZE}.h5'), 'r') as h: X.append(np.asarray(h['images'][:S['n_items'] // 3]))
X = np.concatenate(X); X = X[..., None] if X.ndim == 3 else X
f_img = tf.function(lambda z: img(z, training=False), jit_compile=jit); f_mig = tf.function(lambda z: ft(enc(z, training=False), training=False))
f_tow = tf.function(lambda z: gt(z, training=False)); f_first = tf.function(lambda z: gt(img(z, training=False), training=False), jit_compile=jit)
# ---- correctness against the saved embeddings
l2n = lambda a: a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-12)
Ei = L.load_npz(S['img_dir'] + '/embeddings_float32.npz')['MOLF_DB1'][:16]; Ws = L.load_npz(S['img_dir'] + '/embeddings_float32.npz')['MOLF_DB1'][:16]
c_img = float(np.min(np.sum(l2n(Ei) * f_img(tf.constant(X[:16])).numpy(), 1)))
Em = L.load_npz(md_ + '/embeddings_float32.npz')['MOLF_DB1'][:16]; c_mig = float(np.min(np.sum(l2n(Em) * f_mig(tf.constant(SETS[:16])).numpy(), 1)))
Eg = L.load_npz(md_ + '/image_side_embeddings_float32.npz')['MOLF_DB1'][:16]; c_tow = float(np.min(np.sum(l2n(Eg) * f_tow(tf.constant(l2n(Ei).astype(np.float32))).numpy(), 1)))
def timed(fn, data, name):
    for i in range(S['warmup']): fn(tf.constant(data[i % len(data)][None])).numpy()          # warm-up: excluded
    lat = []
    for i in range(len(data)):
        x = tf.constant(data[i][None]); t = time.perf_counter(); fn(x).numpy(); lat.append(1000 * (time.perf_counter() - t))
    lat = np.asarray(lat); xb = tf.constant(data[:S['batch']]); fn(xb).numpy(); fn(xb).numpy(); tb = []                  # batch warm-up: excluded
    for _ in range(10): t = time.perf_counter(); fn(xb).numpy(); tb.append(1000 * (time.perf_counter() - t) / len(xb))
    return dict(component=name, items=int(len(lat)), ms_mean=float(lat.mean()), ms_median=float(np.median(lat)), ms_std=float(lat.std()), ms_p95=float(np.percentile(lat, 95)),
                ms_min=float(lat.min()), batch_ms_per_item=float(np.median(tb)))
E_img = f_img(tf.constant(X[:64])).numpy().astype(np.float32)
rows = [timed(f_img, X, 'image model (probe image -> embedding)'), timed(f_first, X, 'image model + Siamese image tower (first login)'),
        timed(f_tow, E_img, 'Siamese image tower only'), timed(f_mig, SETS, 'migration: minutiae encoder + Siamese tower (per template)')]
g_ = l2n(np.random.default_rng(0).normal(size=(10000, EMBEDDING_DIM)).astype(np.float32)); q = g_[0].copy()
for i in range(1000): float(q @ g_[i])
t = time.perf_counter()
for i in range(20000): float(q @ g_[i % 10000])
rows.append(dict(component='cosine 1:1 (numpy, CPU)', items=20000, ms_mean=1000 * (time.perf_counter() - t) / 20000))
# ---- complexity (FLOPs per item, batch 1, fp32 graph)
def flops(fn, spec_):
    try:
        from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2
        cf = convert_variables_to_constants_v2(fn.get_concrete_function(spec_)); o = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
        return int(tf.compat.v1.profiler.profile(graph=cf.graph, run_meta=tf.compat.v1.RunMetadata(), cmd='op', options=o).total_float_ops)
    except Exception as exc: return None
cx = None
if precision == 'fp32' and device == 'cpu' and threads == 1:
    cx = [dict(model='image model', parameters=int(img.count_params()), flops_per_item=flops(tf.function(lambda z: img(z, training=False)), tf.TensorSpec((1, IMAGE_SIZE, IMAGE_SIZE, 1), tf.uint8))),
          dict(model='Siamese tower (shared by both sides)', parameters=int(gt.count_params()), flops_per_item=flops(tf.function(lambda z: gt(z, training=False)), tf.TensorSpec((1, M.EMBEDDING_DIM), tf.float32))),
          dict(model='minutiae encoder', parameters=int(enc.count_params()), flops_per_item=flops(tf.function(lambda z: enc(z, training=False)), tf.TensorSpec((1,) + SETS.shape[1:], tf.float32)))]
    for r in cx: r.update(size_mb_fp32=4 * r['parameters'] / 2 ** 20, size_mb_fp16=2 * r['parameters'] / 2 ** 20)
cpu = next((l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')), '')
gpu = ', '.join(d.name for d in tf.config.list_logical_devices('GPU'))
meta = dict(device=device, precision=precision, threads=threads, xla=bool(jit), cpu_model=cpu, gpu_devices=gpu, tensorflow=tf.__version__,
            check_image_model=round(c_img, 5), check_migration=round(c_mig, 5), check_tower=round(c_tow, 5))
json.dump(dict(meta=meta, rows=rows, complexity=cx), open(out, 'w')); print(json.dumps(meta))
"""
(WORK / 'timing_worker.py').write_text(WORKER)
BZ_WORKER = r"""
import os, sys
for v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'): os.environ[v] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import subprocess, shutil, time, ctypes, tempfile, json
from pathlib import Path
sys.path.insert(0, '/content')
""" + 'import numpy as np, pandas as pd, h5py, cv2, struct\nimport matplotlib; matplotlib.use(\'Agg\'); import fpf_lib as L, fpp_lib as P\ncv2.setNumThreads(1)\nws = L.workspace()\nN_PER_DATASET = 100; WARMUP = 10; PAIRS_PER_SET = 700; SINGLE_CALLS = 100; MAX_MINUTIAE = 150; IMAGE_SIZE = 224\nNBIS_REPO = \'https://github.com/mike10004/nbis.git\'; NBIS_COMMIT = \'e2122d4508ab5cc4761fe29caa8a161539c177d8\'      # same pinned NBIS as FPP03\ndef run(cmd, cwd=None):\n    p = subprocess.run(cmd, cwd=cwd, shell=True, text=True, capture_output=True)\n    if p.returncode != 0: raise RuntimeError(f\'{cmd}\\n{p.stdout[-2000:]}\\n{p.stderr[-2000:]}\')\n    return p.stdout\n\n# ---------------------------------------------------------------- FingerJetFX (library built by FPP02, same pinned commit)\nLIB = Path(\'/content/libFJFX.so\'); shutil.copy2(ws[\'templates\'] / \'libFJFX.so\', LIB)\nFMD_ISO = 0x01010001; FMD_BUF = 34 + 256 * 6\n_fjfx = ctypes.CDLL(str(LIB)).fjfx_create_fmd_from_raw\n_fjfx.argtypes = [ctypes.c_void_p, ctypes.c_ushort, ctypes.c_uint, ctypes.c_uint, ctypes.c_uint, ctypes.c_void_p, ctypes.POINTER(ctypes.c_uint)]; _fjfx.restype = ctypes.c_int\ndef extract_iso(gray, ppi):\n    img = np.ascontiguousarray(gray, np.uint8); h, w = img.shape; buf = (ctypes.c_ubyte * FMD_BUF)(); size = ctypes.c_uint(FMD_BUF)\n    err = _fjfx(img.ctypes.data_as(ctypes.c_void_p), int(ppi), h, w, FMD_ISO, ctypes.cast(buf, ctypes.c_void_p), ctypes.byref(size))\n    return int(err), (bytes(buf[:size.value]) if err == 0 else b\'\')\ndef parse_iso(data):\n    pos = 28; n = data[27]; x, y, t, q = [], [], [], []\n    for i in range(n):\n        a, b, ang, qq = struct.unpack(\'>HHBB\', data[pos:pos + 6]); pos += 6; x.append(a & 0x3FFF); y.append(b & 0x3FFF); t.append((360.0 - ang * 360.0 / 256.0) % 360.0); q.append(qq / 100.0)\n    return np.array(x, np.float32), np.array(y, np.float32), np.array(t, np.float32), np.array(q, np.float32)\ndef xyt_text(x, y, theta_img_deg, quality):     # identical to FPP03 write_xyt\n    iso = np.rint((360.0 - np.asarray(theta_img_deg, np.float64)) % 360.0).astype(int) % 360; q = np.rint(100 * np.asarray(quality)).astype(int)\n    return \'\'.join(f\'{int(round(float(x[i])))} {int(round(float(y[i])))} {iso[i]} {q[i]}\\n\' for i in np.argsort(-q, kind=\'stable\')[:MAX_MINUTIAE])\ndef letterbox(image):                            # identical to FPP02\n    h, w = image.shape; s = min(IMAGE_SIZE / w, IMAGE_SIZE / h); nw, nh = max(1, round(w * s)), max(1, round(h * s))\n    r = cv2.resize(image, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)\n    c = np.full((IMAGE_SIZE, IMAGE_SIZE), 255, np.uint8); x0 = (IMAGE_SIZE - nw) // 2; y0 = (IMAGE_SIZE - nh) // 2; c[y0:y0 + nh, x0:x0 + nw] = r; return c\n\n# ---------------------------------------------------------------- BOZORTH3 (same pinned NBIS source and build line as FPP03)\nSRC = Path(\'/content/nbis_src\'); BZ = Path(\'/content/bozorth3_build\'); BZ.mkdir(exist_ok=True); BOZORTH3 = BZ / \'bozorth3\'\nif not BOZORTH3.is_file():\n    if not (SRC / \'.git\').is_dir(): run(f\'git clone -q {NBIS_REPO} {SRC}\')\n    run(f\'git checkout -q {NBIS_COMMIT}\', cwd=SRC)\n    (BZ / \'version.h\').write_text(\'#define VERSION "NBIS bozorth3 (FPP timing build)"\\n\'); (BZ / \'version_stub.c\').write_text(\'#include <stdio.h>\\nvoid getVersion(void){printf("bozorth3\\\\n");}\\n\')\n    run(f\'gcc -O2 -w -I{BZ} -Ibozorth3/include -Icommonnbis/include bozorth3/src/bin/bozorth3/bozorth3.c bozorth3/src/bin/bozorth3/usage.c bozorth3/src/lib/bozorth3/*.c {BZ}/version_stub.c -o {BOZORTH3} -lm\', cwd=SRC)\ndef bozorth3(pairs):\n    with tempfile.NamedTemporaryFile(\'w\', suffix=\'.lis\', delete=False) as f:\n        for a, b in pairs: f.write(f\'{a}\\n{b}\\n\')\n        lis = f.name\n    p = subprocess.run([str(BOZORTH3), \'-m1\', \'-M\', lis], text=True, capture_output=True); os.unlink(lis)\n    if p.returncode != 0: raise RuntimeError(p.stderr[-500:])\n    v = [int(t) for t in p.stdout.split()]; assert len(v) == len(pairs); return np.asarray(v, np.float64)\n\n# ---------------------------------------------------------------- 1) FingerJetFX extraction, xyt, letterbox: per image, on the same originals\nrows_img, check_bytes = [], []\nfor d in L.DATASETS:\n    with h5py.File(L.local_copy(ws[\'originals\'] / f\'{d}_originals_png.h5\'), \'r\') as h:\n        n = len(h[\'png\']); idx = np.linspace(0, n - 1, N_PER_DATASET).astype(int)\n        imgs = [cv2.imdecode(np.asarray(h[\'png\'][i]), cv2.IMREAD_GRAYSCALE) for i in idx]; ppi = h[\'ppi\'][:][idx]\n    with h5py.File(L.local_copy(ws[\'templates\'] / f\'{d}_fingerjetfx_templates.h5\'), \'r\') as h: stored = [bytes(np.asarray(h[\'minutiae/iso_template\'][i])) for i in idx]\n    for i in range(WARMUP): extract_iso(imgs[i % len(imgs)], ppi[i % len(imgs)])\n    for k, (img, pp) in enumerate(zip(imgs, ppi)):\n        t = time.perf_counter(); err, rec = extract_iso(img, pp); fx = 1000 * (time.perf_counter() - t)\n        t = time.perf_counter(); x, y, th, q = parse_iso(rec) if err == 0 else (np.zeros(0),) * 4; txt = xyt_text(x, y, th, q); p_ = Path(f\'/content/_t.xyt\'); p_.write_text(txt); cv = 1000 * (time.perf_counter() - t)\n        t = time.perf_counter(); letterbox(img); lb = 1000 * (time.perf_counter() - t)\n        check_bytes.append(rec == stored[k])\n        rows_img.append({\'dataset\': d, \'image\': int(idx[k]), \'width\': img.shape[1], \'height\': img.shape[0], \'ppi\': int(pp), \'minutiae\': int(len(x)),\n                         \'fingerjetfx_ms\': fx, \'iso_to_xyt_ms\': cv, \'letterbox_224_ms\': lb})\nI = pd.DataFrame(rows_img); same = float(np.mean(check_bytes))\nprint(f\'FingerJetFX output identical to the stored FPP02 templates: {100 * same:.1f} % of {len(check_bytes)} images\')\n\n# ---------------------------------------------------------------- 2) BOZORTH3 on real test pairs (repetition 1), scores checked against FPP03\nrep = L.repetitions(ws)[0]; proto, Pp = P.load_pairs(ws, rep); S03 = P.load_scores(P.kernel_dir(ws, \'FPP03\') / \'runs\' / rep / \'BOZORTH3\')\nXYT = {}\ndef xyt_file(d, i):\n    key = (d, int(i))\n    if key not in XYT:\n        g = H[d][\'minutiae\']; p_ = Path(f\'/content/xyt_timing/{d}/{int(i):06d}.xyt\'); p_.parent.mkdir(parents=True, exist_ok=True)\n        p_.write_text(xyt_text(g[\'x\'][i], g[\'y\'][i], g[\'theta\'][i], g[\'quality\'][i])); XYT[key] = str(p_)\n    return XYT[key]\nH = {d: h5py.File(L.local_copy(ws[\'templates\'] / f\'{d}_fingerjetfx_templates.h5\'), \'r\') for d in L.DATASETS}\nrows_bz, match = [], []\nfor name in (\'DB1_test\', \'DB2_test\', \'FingerDOS_test\'):\n    da, db, _ = P.PAIR_SETS[name]; p = Pp[name]; k = np.linspace(0, len(p[\'label\']) - 1, PAIRS_PER_SET).astype(int)\n    pairs = [(xyt_file(da, p[\'index_a\'][j]), xyt_file(db, p[\'index_b\'][j])) for j in k]\n    bozorth3(pairs[:50])                                                                   # warm-up (file cache)\n    t = time.perf_counter(); sc = bozorth3(pairs); per_pair = 1000 * (time.perf_counter() - t) / len(pairs)\n    single = []\n    for a, b in pairs[:SINGLE_CALLS]: t = time.perf_counter(); bozorth3([(a, b)]); single.append(1000 * (time.perf_counter() - t))\n    ref = S03[name][1][k]; match.append(float(np.mean(sc == ref)))\n    rows_bz.append({\'pair_set\': name, \'pairs\': len(pairs), \'bozorth3_ms_per_pair_amortised\': per_pair, \'bozorth3_ms_one_pair_per_call_mean\': float(np.mean(single)),\n                    \'bozorth3_ms_one_pair_per_call_median\': float(np.median(single)), \'scores_equal_to_FPP03\': float(np.mean(sc == ref))})\nfor h in H.values(): h.close()\nB = pd.DataFrame(rows_bz); print(f\'BOZORTH3 scores identical to the saved FPP03 scores: {100 * np.mean(match):.1f} %\')\n\n' + r"""
I.to_csv(sys.argv[1], index=False); B.to_csv(sys.argv[2], index=False)
json.dump({'fingerjetfx_identical_to_FPP02': same, 'bozorth3_scores_identical_to_FPP03': float(np.mean(match)),
           'cpu_model': next((l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')), '')}, open(sys.argv[3], 'w'))
"""
(WORK / 'bozorth3_worker.py').write_text(BZ_WORKER); print('workers written to', WORK)

## 3. Main time comparison: image-embedding matching (ours) vs FingerJetFX + BOZORTH3 matching — measured in this session (cached)

In [ ]:
def hw_label(cfg):
    if cfg['device'] == 'gpu': return f"{GPU_NAME} {cfg['precision']}" + (' + XLA' if cfg['jit'] == 'xla' else '')
    return f"CPU {cfg['threads']} thread{'s' if cfg['threads'] > 1 else ''}" + (' + XLA' if cfg['jit'] == 'xla' else '')
raw_csv, bzi_csv, bzb_csv = TDIR / 'raw.csv', TDIR / 'fingerjetfx_per_image.csv', TDIR / 'bozorth3_per_set.csv'
if RERUN_TIMING or not (raw_csv.is_file() and bzi_csv.is_file() and bzb_csv.is_file()):
    if not GPU: print('WARNING: no GPU in this runtime — accelerator rows will be missing; use a T4 runtime')
    RAW = []
    for cfg in CONFIGS:
        for k in range(MEASURE_REPEATS):
            out = WORK / f"t_{cfg['device']}_{cfg['precision']}_{cfg['threads']}_{cfg['jit']}_{k}.json"; t0 = time.time()
            r = subprocess.run([sys.executable, str(WORK / 'timing_worker.py'), cfg['device'], cfg['precision'], str(cfg['threads']), str(out), cfg['jit']], capture_output=True, text=True)
            if r.returncode != 0: print(r.stdout[-1500:], r.stderr[-3000:]); raise RuntimeError(f'timing failed: {cfg}')
            d = json.load(open(out)); m = d['meta']
            if m['check_image_model'] < (0.99 if cfg['precision'] == 'fp16' else 0.999): raise RuntimeError(f'image model check failed: {m}')
            for row in d['rows']: RAW.append({**cfg, 'hardware': hw_label(cfg), 'measure': k, 'check_image_model': m['check_image_model'], 'cpu_model': m['cpu_model'], 'tensorflow': m['tensorflow'], **row})
            print(f"{hw_label(cfg)} measure {k + 1}: ok ({time.time() - t0:.0f} s) | check {m['check_image_model']}")
    RAW = pd.DataFrame(RAW); BZI, BZB = [], []
    for k in range(MEASURE_REPEATS):
        fi, fb, fc = WORK / f'fjfx_{k}.csv', WORK / f'bz_{k}.csv', WORK / f'chk_{k}.json'
        r = subprocess.run([sys.executable, str(WORK / 'bozorth3_worker.py'), str(fi), str(fb), str(fc)], capture_output=True, text=True)
        if r.returncode != 0: print(r.stdout[-1500:], r.stderr[-3000:]); raise RuntimeError('FingerJetFX / BOZORTH3 timing failed')
        ch = json.load(open(fc))
        if min(ch['fingerjetfx_identical_to_FPP02'], ch['bozorth3_scores_identical_to_FPP03']) < 0.999: raise RuntimeError(f'FingerJetFX / BOZORTH3 outputs differ: {ch}')
        BZI.append(pd.read_csv(fi).assign(measure=k)); BZB.append(pd.read_csv(fb).assign(measure=k)); print(f'FingerJetFX + BOZORTH3 measure {k + 1}: ok | {ch}')
    BZI = pd.concat(BZI, ignore_index=True); BZB = pd.concat(BZB, ignore_index=True)
    RAW.to_csv(raw_csv, index=False); BZI.to_csv(bzi_csv, index=False); BZB.to_csv(bzb_csv, index=False)
else:
    RAW, BZI, BZB = pd.read_csv(raw_csv), pd.read_csv(bzi_csv), pd.read_csv(bzb_csv); print('timing loaded from', L.rel(ws, TDIR), '(set RERUN_TIMING = True to measure again)')
IM = 'image model (probe image -> embedding)'
cos_ms = float(RAW[RAW.component.str.startswith('cosine')].ms_mean.mean()); lb = BZI.groupby('measure').letterbox_224_ms.mean()
fx = BZI.groupby('measure').apply(lambda s: s.fingerjetfx_ms.mean() + s.iso_to_xyt_ms.mean()); bz = BZB.groupby('measure').bozorth3_ms_per_pair_amortised.mean(); old = fx + bz
f2 = lambda s: f'{s.mean():.2f} ± {s.std():.2f}'
rows = [{'method': 'FingerJetFX + BOZORTH3 (old system)', 'hardware': 'CPU 1 thread', 'template_creation_ms': f2(fx), 'matching_1to1_ms': f'{bz.mean():.3f} ± {bz.std():.3f}',
         'probe_to_decision_ms': f2(old), 'speed_up_vs_old': 1.0, '_ms': float(old.mean())}]
for hw in RAW.hardware.unique():
    im = RAW[(RAW.hardware == hw) & (RAW.component == IM)].groupby('measure').ms_mean.mean(); tc = im + float(lb.mean()); tot = tc + cos_ms
    rows.append({'method': 'Ours: letterbox + image model + cosine', 'hardware': hw, 'template_creation_ms': f2(tc), 'matching_1to1_ms': f'{cos_ms:.4f}', 'probe_to_decision_ms': f2(tot),
                 'speed_up_vs_old': round(float(old.mean() / tot.mean()), 2), '_ms': float(tot.mean())})
TIME = pd.DataFrame(rows)
print('MAIN TIME COMPARISON — daily 1:1 login after migration: image-embedding matching vs FingerJetFX + BOZORTH3 matching (same session, same machine; warm-up excluded):'); display(TIME.drop(columns='_ms'))
print('Hardware:', RAW.cpu_model.iloc[0], '|', os.cpu_count(), 'cores | GPU:', GPU_NAME, '| TensorFlow', RAW.tensorflow.iloc[0])

## 4. Main accuracy comparison (identical pairs, frozen validation-EER threshold, mean ± std over 3 repetitions)

In [ ]:
def rows_of(label, run_dir_by_rep):
    out = []
    for r, rd in run_dir_by_rep.items():
        rd = Path(rd); out += P.reference_rows(label, rd.parents[2], arm=rd.name, reps=[r])          # rd = <kernel>/runs/<rep>/<arm>
    df = pd.DataFrame(out)
    if len(df): df['kernel'] = label
    return df
DIRS = {IMG_LABEL: {r: p_.parent for r, p_ in img_files.items()},
        BOZ: {r: P.kernel_dir(ws, 'FPP03') / 'runs' / r / 'BOZORTH3' for r in REPS},
        MIN_LABEL: {r: p_.parent for r, p_ in winner('FPP09')[0].items()},
        MIG_LABEL: {r: p_.parent for r, p_ in mig_files.items()}}
ACC = pd.concat([rows_of(k, v) for k, v in DIRS.items()], ignore_index=True); MAIN = list(DIRS)
SETS_ = ['MOLF validation (balanced)', 'MOLF test (balanced)', 'DB1_test', 'DB2_test', 'DB1xDB2_test', 'FingerDOS_test']
acc_all, acc_num = P.paper_table(ACC, MAIN, sets=SETS_)
ACC_TABLE = P.wide_table(acc_num, MAIN, sets=['MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test'], metrics_=('roc_auc', 'eer', 'fmr', 'fnmr', 'f1_macro', 'accuracy'))
print('MAIN ACCURACY COMPARISON (test sets; reported):'); display(ACC_TABLE)
VAL_TABLE = P.validation_table(ACC, MAIN); print('Validation (the set every decision was made on):'); display(VAL_TABLE)
PAIRED = P.paired_differences(ACC, IMG_LABEL, [BOZ, MIN_LABEL])
key = PAIRED[PAIRED.metric.isin(['eer', 'roc_auc', 'f1_macro']) & PAIRED.set.isin(['MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test'])]
print('Paired differences, ours (image vs image) minus reference, in points (negative EER = ours better):'); display(key.round(3))

## 5. Progression of every kernel's winner (validation decides; test and FingerDOS reported)

In [ ]:
STEPS = [('image', 'FPP04', None), ('image', 'FPP04C', None), ('image', 'FPP04L', None), ('image', 'FPP05', None),
         ('minutiae', 'FPP06', None), ('minutiae', 'FPP07', None), ('minutiae', 'FPP08', None), ('minutiae', 'FPP09', None),
         ('migration (first login)', 'FPP10', 'two_tower'), ('migration (first login)', 'FPP11', 'best_plus_mm'), ('migration (first login)', 'FPP12', 'siamese'), ('migration (first login)', 'FPP13', None)]
PR = []; ORDER = []
for stage, k, arm in STEPS:
    if k not in P.FPP_KERNELS: continue
    if arm is None:
        files, mf = winner(k)
        if files is None: print(k, ': no winner manifest -> skipped'); continue
        dirs = {r: p_.parent for r, p_ in files.items()}; arm_ = mf.get('winning_arm', next(iter(dirs.values())).name)
    else:
        dirs = {r: P.kernel_dir(ws, k) / 'runs' / r / arm for r in REPS}; arm_ = arm
        if not all((d / 'pair_scores.npz').is_file() for d in dirs.values()): print(k, arm, ': missing -> skipped'); continue
    label = f'{k} {arm_} [{stage}]'; df = rows_of(label, dirs)
    if len(df): PR.append(df); ORDER.append(label)
PR = pd.concat(PR, ignore_index=True) if PR else pd.DataFrame()
if len(PR):
    pr_all, pr_num = P.paper_table(PR, ORDER, sets=['MOLF validation (balanced)', 'MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test'])
    PROG = P.wide_table(pr_num, ORDER, sets=['MOLF validation (balanced)', 'MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test'], metrics_=('eer', 'roc_auc', 'f1_macro'))
    print('PROGRESSION — EER / AUC / F1 (%), mean ± std:'); display(PROG)
else: PROG = pd.DataFrame()

## 6. Reported from FPP14: first login, migration, complexity

In [ ]:
K14 = P.kernel_dir(ws, 'FPP14'); F14 = {n_: K14 / f'FPP14_{n_}.csv' for n_ in ('scenarios', 'migration', 'complexity')}
T14 = {n_: (pd.read_csv(p_) if p_.is_file() else pd.DataFrame()) for n_, p_ in F14.items()}
FIRST = T14['scenarios'][T14['scenarios'].scenario.astype(str).str.startswith('First login')] if len(T14['scenarios']) else pd.DataFrame()
for n_, df in (('First login after migration (FPP14)', FIRST), ('Migration, once per template (FPP14)', T14['migration']), ('Complexity (FPP14)', T14['complexity'])):
    print(n_ + ':'); display(df)

## 7. Figures

In [ ]:
MR = {lab: [Path(d) for d in DIRS[lab].values() if (Path(d) / 'pair_scores.npz').is_file()] for lab in MAIN}; MR = {k_: v_ for k_, v_ in MR.items() if v_}
P.standard_figures(FIG, MR, sets=P.PAPER_SETS)
P.fig_metric_bars(acc_num, [m_ for m_ in MAIN if m_ in set(ACC.kernel)], FIG / 'final_accuracy_bars.png', sets=['MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test'], title='Final accuracy on the test sets (mean ± std over 3 repetitions)')
fig, ax = plt.subplots(figsize=(9, 3.8)); lab_ = [f"{m}\n[{h}]" for m, h in zip(TIME.method, TIME.hardware)]
ax.barh(lab_, TIME._ms, color=['0.55'] + [P.COLORS[0]] * (len(TIME) - 1)); ax.set_xscale('log'); ax.invert_yaxis()
for y_, (v_, s_) in enumerate(zip(TIME._ms, TIME.speed_up_vs_old)): ax.text(v_, y_, f' {v_:.1f} ms ({s_}×)', va='center', fontsize=8)
ax.set(xlabel='probe to decision, 1:1 (ms, log scale; warm-up excluded)', title='Daily login: image-embedding matching (ours) vs FingerJetFX + BOZORTH3 matching'); ax.grid(alpha=.3, axis='x')
P._save(fig, FIG / 'final_time_comparison.png')
if len(PR):
    v_ = PR[PR.set == 'MOLF validation (balanced)'].groupby('kernel').eer.agg(['mean', 'std']).reindex(ORDER) * 100
    fig, ax = plt.subplots(figsize=(11, 4)); cols = {'image': 'C0', 'minutiae': 'C1', 'migration (first login)': 'C2'}
    ax.bar(range(len(ORDER)), v_['mean'], yerr=v_['std'], capsize=3, color=[cols[o.split('[')[-1][:-1]] for o in ORDER])
    ax.set_xticks(range(len(ORDER))); ax.set_xticklabels([o.split(' [')[0] for o in ORDER], rotation=35, ha='right', fontsize=7)
    ax.set(ylabel='validation EER, source-balanced (%)', title='Winner of every kernel (blue: image, orange: minutiae, green: first-login migration)'); ax.grid(alpha=.3, axis='y')
    P._save(fig, FIG / 'progression.png')
print('figures in', FIG)

## 8. Workbook, CSVs and Markdown summary

In [ ]:
SHEETS = {'Time (main)': TIME.drop(columns='_ms'), 'Accuracy (main, test)': ACC_TABLE, 'Accuracy (validation)': VAL_TABLE, 'Accuracy (all sets)': acc_all, 'Paired vs BOZORTH3': PAIRED,
          'Progression': PROG, 'First login (FPP14)': FIRST, 'Migration (FPP14)': T14['migration'], 'Complexity (FPP14)': T14['complexity'],
          'Timing raw': RAW, 'FingerJetFX per image': BZI, 'BOZORTH3 per set': BZB, 'Results (per rep)': P.to_percent(pd.concat([ACC, PR], ignore_index=True)),
          'Config': pd.DataFrame([{k: json.dumps(v_, default=str) for k, v_ in CONFIG.items()}]).T.reset_index()}
L.write_workbook(KDIR / 'FPP15_Final_Report.xlsx', SHEETS)
for n_, df in (('time_main', TIME.drop(columns='_ms')), ('accuracy_main_test', ACC_TABLE), ('accuracy_validation', VAL_TABLE), ('paired_vs_bozorth3', PAIRED), ('progression', PROG)): df.to_csv(KDIR / f'FPP15_{n_}.csv', index=False)
def md_table(df):
    df = df.reset_index(drop=True) if not isinstance(df.columns, pd.MultiIndex) else df.copy()
    if isinstance(df.columns, pd.MultiIndex): df.columns = [' '.join(str(x) for x in col if str(x) != '') for col in df.columns]; df = df.reset_index()
    h = '| ' + ' | '.join(map(str, df.columns)) + ' |\n|' + '---|' * len(df.columns) + '\n'
    return h + ''.join('| ' + ' | '.join(str(v_) for v_ in r) + ' |\n' for r in df.astype(str).values)
best = TIME.iloc[1:].sort_values('_ms').iloc[0] if len(TIME) > 1 else None
lines = ['# FPP15 — Final report', '', f"Hardware: {RAW.cpu_model.iloc[0]}, {os.cpu_count()} cores; GPU {GPU_NAME}; TensorFlow {RAW.tensorflow.iloc[0]}. Timing: one item per call, warm-up excluded, 3 measures.", '',
         '## Time — daily 1:1 login: image-embedding matching (ours) vs FingerJetFX + BOZORTH3 matching', '', md_table(TIME.drop(columns='_ms')), '']
if best is not None: lines += [f"Fastest configuration of ours: **{best.hardware}**, {best._ms:.2f} ms probe to decision = **{best.speed_up_vs_old}×** the old system.", '']
lines += ['## Accuracy — test sets', '', md_table(ACC_TABLE), '', '## Accuracy — validation', '', md_table(VAL_TABLE), '']
if len(PROG): lines += ['## Progression of the kernel winners', '', md_table(PROG), '']
(KDIR / 'FPP15_Final_Report.md').write_text('\n'.join(lines), encoding='utf-8'); print('report:', L.rel(ws, KDIR / 'FPP15_Final_Report.md'), '| workbook: FPP15_Final_Report.xlsx')